Databricks notebook: nb_gold_dim_tariff
Source : ev_catalog.silver.sl_tariffs
Sink   : ev_catalog.gold.dim_tariff  (Delta, external)
Type   : SCD Type 2 — one row per tariff version (effective_from / effective_to)
Grain  : One row per tariff version window

Silver sl_tariffs columns:
  tariff_id, version (INT), peak_offpeak, rate_per_kwh (DOUBLE),
  effective_from (STRING), effective_to (STRING), is_current (BOOLEAN),
  created_at_utc, created_at_aest, updated_at_utc, updated_at_aest,
  sl_ingested_at, sl_batch_id

In [ ]:
# ── Cell 1: imports and config ────────────────────────────────────────────────
from pyspark.sql import functions as F
from delta.tables import DeltaTable

GOLD_TABLE = "ev_catalog.gold.dim_tariff"
GOLD_PATH  = "abfss://gold@evdatalakedev2.dfs.core.windows.net/dimensions/dim_tariff/"

dbutils.widgets.text("p_run_date", "", "Run date YYYY-MM-DD (blank = full load)")
run_date = dbutils.widgets.get("p_run_date").strip()

In [ ]:
# ── Cell 2: read Silver source ─────────────────────────────────────────────────
df_silver = spark.table("ev_catalog.silver.sl_tariffs")

In [ ]:
# ── Cell 3: build dimension ───────────────────────────────────────────────────
# Silver gives us rate_per_kwh as DOUBLE, version as INT, is_current as BOOLEAN.
# effective_from/effective_to are STRING — cast to DATE here.
# created_at/updated_at raw strings are dropped by Silver; use _utc/_aest variants.

df_dim = (
    df_silver.select(
        F.col("tariff_id")                    .alias("tariff_key"),
        F.col("version"),
        F.col("peak_offpeak"),
        F.col("rate_per_kwh"),
        F.to_date(F.col("effective_from"))    .alias("effective_from"),
        F.to_date(F.col("effective_to"))      .alias("effective_to"),
        F.col("is_current"),
        F.col("created_at_utc"),
        F.col("created_at_aest"),
        F.col("updated_at_utc"),
        F.col("updated_at_aest"),
        F.current_timestamp()                 .alias("gold_updated_at"),
    )
)

print(f"dim_tariff rows: {df_dim.count()}")
df_dim.show(3, truncate=80)

In [ ]:
# ── Cell 4: CREATE TABLE IF NOT EXISTS ────────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
    tariff_key       STRING    NOT NULL,
    version          INT,
    peak_offpeak     STRING,
    rate_per_kwh     DOUBLE,
    effective_from   DATE,
    effective_to     DATE,
    is_current       BOOLEAN,
    created_at_utc   TIMESTAMP,
    created_at_aest  TIMESTAMP,
    updated_at_utc   TIMESTAMP,
    updated_at_aest  TIMESTAMP,
    gold_updated_at  TIMESTAMP
)
USING DELTA
LOCATION '{GOLD_PATH}'
""")

In [ ]:
# ── Cell 5: MERGE ─────────────────────────────────────────────────────────────
# Merge on tariff_id + effective_from — each version window is a distinct row.
tgt = DeltaTable.forName(spark, GOLD_TABLE)
(
    tgt.alias("t")
       .merge(
           df_dim.alias("s"),
           "t.tariff_key = s.tariff_key AND t.effective_from = s.effective_from"
       )
       .whenMatchedUpdateAll()
       .whenNotMatchedInsertAll()
       .execute()
)
print(f"MERGE done: {GOLD_TABLE}")

In [ ]:
# ── Cell 6: verify ────────────────────────────────────────────────────────────
spark.sql(f"""
SELECT tariff_key, peak_offpeak, rate_per_kwh, effective_from, effective_to, is_current
FROM   {GOLD_TABLE}
ORDER  BY tariff_key, effective_from DESC
""").display()